# Lesson 2: Tool Calling

## Setup

Run the first code cell before anything else: it adds the repo root to `sys.path` and loads `ANTHROPIC_API_KEY` via `helper` (including from `.env`). Then run the `nest_asyncio` cell and the model `Settings` cell.

In [3]:
import sys
from pathlib import Path

_root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "helper.py").is_file() and (p / "utils.py").is_file()
    ),
    None,
)
if _root is None:
    raise RuntimeError(
        "Could not find project root (helper.py + utils.py). "
        "Open the training repo folder or run from a lesson subfolder inside it."
    )
sys.path.insert(0, str(_root))

from helper import get_anthropic_api_key
ANTHROPIC_API_KEY = get_anthropic_api_key()

In [4]:
import nest_asyncio
nest_asyncio.apply()

In [5]:
from llama_index.core import Settings
from llama_index.llms.anthropic import Anthropic
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

Settings.llm = Anthropic(
    model="claude-sonnet-4-6", api_key=ANTHROPIC_API_KEY
)
Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

/Users/trentoncreamer/Crypto/DataCore Labs/Training/building-agentic-rag-with-llamaindex/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6188.35it/s]
BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 1. Define a Simple Tool

In [6]:
from llama_index.core.tools import FunctionTool

def add(x: int, y: int) -> int:
    """Adds two integers together."""
    return x + y

def mystery(x: int, y: int) -> int: 
    """Mystery function that operates on top of two numbers."""
    return (x + y) * (x + y)


add_tool = FunctionTool.from_defaults(fn=add)
mystery_tool = FunctionTool.from_defaults(fn=mystery)

In [ ]:
from llama_index.llms.anthropic import Anthropic

llm = Anthropic(model="claude-sonnet-4-6", api_key=ANTHROPIC_API_KEY)
# llm decides what tool to call and calls the tool itself
response = llm.predict_and_call(
    [add_tool, mystery_tool], 
    "Tell me the output of the mystery function on 2 and 9", 
    verbose=True
)
print(str(response))

=== Calling Function ===
Calling function: mystery with args: {"x": 2, "y": 9}
=== Function Output ===
121
121


## 2. Define an Auto-Retrieval Tool

### Load Data

To download this paper, below is the needed code:

#!wget "https://openreview.net/pdf?id=VtmBAGCN7o" -O metagpt.pdf

**Note**: The pdf file is included with this lesson. To access it, go to the `File` menu and select`Open...`.

In [9]:
from llama_index.core import SimpleDirectoryReader
# load documents
documents = SimpleDirectoryReader(input_files=["metagpt.pdf"]).load_data()

In [10]:
from llama_index.core.node_parser import SentenceSplitter
splitter = SentenceSplitter(chunk_size=1024)
nodes = splitter.get_nodes_from_documents(documents)

In [12]:
print(nodes[0].get_content(metadata_mode="all"))

file_path: metagpt.pdf
file_name: metagpt.pdf
file_type: application/pdf
file_size: 16911937
creation_date: 2026-03-28
last_modified_date: 2026-03-28

%PDF-1.5
%
718 0 obj
<< /Linearized 1 /L 16911937 /H [ 3577 512 ] /O 722 /E 96497 /N 29 /T 16906639 >>
endobj
                                                                                                     
719 0 obj
<< /Type /XRef /Length 124 /Filter /FlateDecode /DecodeParms << /Columns 5 /Predictor 12 >> /W [ 1 3 1 ] /Index [ 718 415 ] /Info 134 0 R /Root 720 0 R /Size 1133 /Prev 16906640              /ID [<fd88fb1aa59ef1255062b67f2a441337><2be470bb3c12d18e76bfc564a5134d18>] >>
stream
xcbd`g`b``8	"yٍ A$. RXLHs [DUHgo efEBA$'ٙ`{GQR}a$w(9HyQrx 
endstream
endobj
                                                                                                                                                                                                                                                                 

In [11]:
from llama_index.core import VectorStoreIndex

vector_index = VectorStoreIndex(nodes, show_progress=True)
query_engine = vector_index.as_query_engine(similarity_top_k=2)

Generating embeddings: 100%|██████████| 737/737 [00:13<00:00, 55.29it/s]


In [13]:
from llama_index.core.vector_stores import MetadataFilters

# Filter based on metadata 
query_engine = vector_index.as_query_engine(
    similarity_top_k=2,
    filters=MetadataFilters.from_dicts(
        [
            {"key": "page_label", "value": "2"}
        ]
    )
)

response = query_engine.query(
    "What are some high-level results of MetaGPT?", 
)

In [ ]:
print(str(response))

In [ ]:
for n in response.source_nodes:
    print(n.metadata)

### Define the Auto-Retrieval Tool

In [ ]:
from typing import List
from llama_index.core.vector_stores import FilterCondition


def vector_query(
    query: str, 
    page_numbers: List[str]
) -> str:
    """Perform a vector search over an index.
    
    query (str): the string query to be embedded.
    page_numbers (List[str]): Filter by set of pages. Leave BLANK if we want to perform a vector search
        over all pages. Otherwise, filter by the set of specified pages.
    
    """

    metadata_dicts = [
        {"key": "page_label", "value": p} for p in page_numbers
    ]
    
    query_engine = vector_index.as_query_engine(
        similarity_top_k=2,
        filters=MetadataFilters.from_dicts(
            metadata_dicts,
            condition=FilterCondition.OR
        )
    )
    response = query_engine.query(query)
    return response
    

vector_query_tool = FunctionTool.from_defaults(
    name="vector_tool",
    fn=vector_query
)

In [ ]:
llm = Anthropic(
    model="claude-sonnet-4-6", temperature=0, api_key=ANTHROPIC_API_KEY
)
response = llm.predict_and_call(
    [vector_query_tool], 
    "What are the high-level results of MetaGPT as described on page 2?", 
    verbose=True
)

In [ ]:
for n in response.source_nodes:
    print(n.metadata)

## Let's add some other tools!

In [ ]:
from llama_index.core import SummaryIndex
from llama_index.core.tools import QueryEngineTool

summary_index = SummaryIndex(nodes)
summary_query_engine = summary_index.as_query_engine(
    response_mode="tree_summarize",
    use_async=False,  # avoid nested-async issues in Jupyter
)
summary_tool = QueryEngineTool.from_defaults(
    name="summary_tool",
    query_engine=summary_query_engine,
    description=(
        "Useful if you want to get a summary of MetaGPT"
    ),
)

In [ ]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What are the MetaGPT comparisons with ChatDev described on page 8?", 
    verbose=True
)

In [ ]:
for n in response.source_nodes:
    print(n.metadata)

In [ ]:
response = llm.predict_and_call(
    [vector_query_tool, summary_tool], 
    "What is a summary of the paper?", 
    verbose=True
)